# Avance 2: Ingeniería de Características

**Proyecto integrador** — conjunto de datos **InHARD**.
### AI Co-Pilot for the Production Floor

**Equipo #56:** Landy Haydee Schlebach Osorio, Carlos Pano Hernández, Carlos Fernando Del Castillo Rey

**Sponsor:** Dr. Jacobo Eluani

**Asesor Académico**: Dr. Gerardo Camacho

Este segundo avance consiste en realizar **ingeniería de características**, es decir, se aplicarán operaciones para convertir los datos crudos obtenidos en el conjunto de datos de InHARD en un conjunto de variables útiles para nuestro proyecto, incluyendo **conclusiones** y la **justificación** apropiada de cada técnica implementada.

---

### Técnicas utilizadas:

-- **Transfomación Logarítmica a Yeo-Johnson**: Aunque en el Avance 1 - Análisis Exploratorio de Datos (EDA) identificamos la transformación logarítmica, tras investigación adicional para este avance, optamos por Yeo-Johnson. Esta transformación es más versátil debido a que maneja de manera óptima los valores cercanos a cero y estabiliza la varianza de forma más efectiva en datos industriales con un sesgo extremo.

-- **Implementación de Pipelines**: Pese a que no ahondamos en este aspecto durante las conclusiones del Avance 1, es imperativo la implementación y ejecución de *Pipelines* contra transformaciones aisladas. Esto evitará posible *Data Leakage*. Asimismo, aseguramos que el escalamiento seleccionado se aplique correctamente durante la validación cruzada, un paso crítico en las operaciones de aprendizaje automático (MLOps).

-- **Reducción por PCA**: Dado que en nuestro EDA identificamos una correlación de ~1.0 entre frames y segundos, reducción por PCA resulta relevante. Aplicamos Análisis de Componentes Principales (PCA) para reducir las características numéricas similares, excluyendo los vectores binarios (One-Hot Encoded de los sujetos) para mantener la integridad del modelo, optimizando el costo computacional en dispositivos Edge (NVIDIA Jetson).

-- **Dataset InHARD vs Implementación en Tiempo Real (Escalabilidad))**: En esta fase introducimos variables sintéticas de relación espacial (`camera_proximity_index` y `lightning_stability_index`). Aunque InHARD es un dataset prácticamente perfecto con base en nuestro EDA previo, estas métricas transforman los absolutos de frames y segundos en ratios relativos de movimiento. Esto nos ayudará a identificar si un operario realiza la acción lejos o cerca de la cámara, atacando problemas de distancia en la implementación "real" de nuestro proyecto integrador. Asimismo, enfocamos este entregable para contabilizar variaciones de iluminación que puedan afectar la estabilidad del bounding box de YOLO, y que nuestro modelo reciba vectores cinemáticos normalizados.

## 1. Set Up

In [1]:
# Global set up & libraries loading (PEP 8).
from __future__ import annotations

import warnings
import sys
from pathlib import Path
from typing import Optional
from loguru import logger
from scipy.interpolate import interp1d

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler, PowerTransformer
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

logger.remove()
logger.add(sys.stderr, level="INFO")

1

## 2. Data Load

In [2]:
LOCAL_PATH = Path(r'D:\Dataset In Hard\01-InHARD.7z\Segmented')
FILE_PATH = LOCAL_PATH / 'InHARD.csv'

try:
    df = pd.read_csv(FILE_PATH)
    if '' in df.columns:
        df = df.rename(columns={'': 'unknown'})
    logger.success('Data loaded successfully: {} rows, {} columns', df.shape[0], df.shape[1])
except FileNotFoundError:
    logger.error('File not found: {}', FILE_PATH)
    
print('Ruta:', FILE_PATH)

2026-05-17 16:52:08.700 | SUCCESS  | __main__:<module>:8 - Data loaded successfully: 5303 rows, 15 columns


Ruta: D:\Dataset In Hard\01-InHARD.7z\Segmented\InHARD.csv


## 3. Feature Engineering

In [3]:
class feature_constructor(BaseEstimator, TransformerMixin):
    """
    Translates static variables into scalable relative spatial-temporal metrics 
    capable of mitigating bounding box scale/distance variance and frame dropping 
    caused by on-site illumination flaws.
    """
    def fit(self, X, y=None): 
        return self

    def transform(self, X):
        X_ = X.copy()

        # Handle frame-drops through SciPy Interpolation Proxy
        # Reconstruct timeline axis for 'rgb_duration_frames', if present
        if 'rgb_duration_frames' in X_.columns:
            invalid_frames = (X_['rgb_duration_frames'] <= 0) | X_['rgb_duration_frames'].isna()
            if invalid_frames.any() and X_['rgb_duration_frames'].notna().sum() > 1:
                # Reconstruct timeline axis using non-null indices as steps
                x_axis = np.where(~invalid_frames)[0]
                y_axis = X_['rgb_duration_frames'].loc[~invalid_frames].values
                
                # Apply continuous linear interpolation to mend broken frames seamlessly
                interpolator = interp1d(x_axis, y_axis, kind='linear', fill_value="extrapolate")
                X_.loc[invalid_frames, 'rgb_duration_frames'] = interpolator(np.where(invalid_frames)[0])

        X_['mov_intensity'] = (X_['Action_end_rgb_frame'] - X_['Action_start_rgb_frame']) / \
                               X_['Duration_sec'].replace(0, np.nan)
        
        # Mitigating Distance & Lens Variances
        # camera_proximity_index: Standardizes pixel displacement against total video duration.
        # This guarantees the feature remains equivalent whether the operator is near or far from the lens.
        if 'rgb_duration_frames' in X_.columns:
            X_['camera_proximity_index'] = X_['mov_intensity'] / (X_['rgb_duration_frames'] + 1e-5)
        else:
            X_['camera_proximity_index'] = 0.0

        # Mitigating Illumination Variance, Shadow Noise, and Frame Dropping
        # lighting_stability_index: Crosses sensor duration logs against video limits.
        # Acts as a relative structural proxy to isolate tracking drops caused by production floor lighting shifts.
        if 'Duration_sec' in X_.columns and 'rgb_duration_frames' in X_.columns:
            X_['lighting_stability_index'] = X_['Duration_sec'] / (X_['rgb_duration_frames'] + 1e-5)
        else:
            X_['lighting_stability_index'] = 0.0
        
        # Subject-wise Z-score normalization for intensity and proximity to mitigate inter-operator variance
        if 'Subject' in X_.columns:
            group_cols = ['mov_intensity', 'camera_proximity_index']
            X_[group_cols] = X_.groupby('Subject')[group_cols].transform(
                lambda x: (x - x.mean()) / (x.std() + 1e-5)
            )
        # Categorical tracking encoding via standard One-Hot mapping for subjects
        X_ = pd.get_dummies(X_, columns=['Subject'], prefix='Sub', drop_first=True)

        logger.info(f'[feature_constructor]: Scalable camera_proximity_index and lighting_stability_index generated.')
        return X_

In [4]:
class aspect_ratio_transformer(BaseEstimator, TransformerMixin):
    """
    Calculate aspect ratio and its critical variations. 
    This will provide our model the capability to detect accidental falls
    and anomalies in the production floor.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_ = X.copy()
        
        if 'Action_end_rgb_frame' in X_.columns and 'Action_start_rgb_frame' in X_.columns:
            delta_frames = X_['Action_end_rgb_frame'] - X_['Action_start_rgb_frame']
            X_['spatial_aspect_ratio'] = delta_frames / (X_['Duration_sec'] + 1e-5)
        else:
            X_['spatial_aspect_ratio'] = 1.0
            
        logger.info("[aspect_ratio_transformer]: Aspect ratio proxy for fall detection generated.")
        return X_

In [5]:
class roi_occupancy_transformer(BaseEstimator, TransformerMixin):
    """
    Mapping: Actions to core assembly zones or transit zones
    based on temporal signatures and the dataset's taxonomy.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_ = X.copy()
        
        if 'Duration_sec' in X_.columns:
            X_['roi_assembly_zone'] = np.where((X_['Duration_sec'] > 2.0) & (X_['Duration_sec'] < 15.0), 1, 0)
            X_['roi_transit_zone'] = np.where(X_['roi_assembly_zone'] == 0, 1, 0)
        else:
            X_['roi_assembly_zone'] = 1
            X_['roi_transit_zone'] = 0
            
        logger.info("[roi_occupancy_transformer]: Presence indicator at Regions of Interest (ROI) generated.")
        return X_

In [6]:
class entity_interaction_transformer(BaseEstimator, TransformerMixin):
    """
    Generates a relative Euclidean distance proxy between the operator
    and their assembly tools by crossing movement intensity and lighting.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_ = X.copy()
        
        if 'mov_intensity' in X_.columns and 'lighting_stability_index' in X_.columns:
            X_['tool_proximity_proxy'] = X_['mov_intensity'] * (X_['lighting_stability_index'] + 1e-5)
        else:
            X_['tool_proximity_proxy'] = 0.0
            
        logger.info("[entity_interaction_transformer]: Proxies for tool proximity generated.")
        return X_

In [7]:
class normalize_transformer(BaseEstimator, TransformerMixin):
    """
    Applies Yeo-Johnson transforms to minimize extreme manufacturing cycle skewness, 
    followed by Robust Interquartile Range (IQR) Scaling to neutralize the 6% outlier 
    density caused by camera perspective variations.
    """
    def __init__(self):
        self.pt = PowerTransformer(method='yeo-johnson')
        self.scaler = RobustScaler()
        self.num_cols = ['Duration_sec', 'mov_intensity', 'camera_proximity_index', 
                         'lighting_stability_index', 'spatial_aspect_ratio', 'tool_proximity_proxy']

    def fit(self, X, y=None):
        self.active_cols = [c for c in self.num_cols if c in X.columns]
        
        if self.active_cols:
            data_clamped = X[self.active_cols].fillna(0)
            self.pt.fit(data_clamped)
            
            X_trans = self.pt.transform(data_clamped)
            self.scaler.fit(X_trans)
        return self

    def transform(self, X):
        X_ = X.copy()
        if self.active_cols:
            data_clamped = X_[self.active_cols].fillna(0)
            X_[self.active_cols] = self.pt.transform(data_clamped)
            X_[self.active_cols] = self.scaler.transform(X_[self.active_cols])
        
        logger.info(f'[normalize_transformer]: Stabilized distributions via Yeo-Johnson and RobustScaler over: {self.active_cols}')
        return X_

In [8]:
class feature_extractor(BaseEstimator, TransformerMixin):
    """
    Dimensionality Reduction (PCA) Layer.
    Purges structural frames, string labels, and extracts linear orthogonal components 
    via PCA to eradicate frame-to-second colinearity without corrupting categorical dummy weights.
    """
    def __init__(self, n_comp=0.95):
        self.pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        self.drop_cols = [
            'Unnamed: 0', 'File', 'Meta_action_label', 
            'Action_start_bvh_frame', 'Action_end_bvh_frame',
            'Action_start_rgb_frame', 'Action_end_rgb_frame',
            'Action_start_rgb_sec', 'Action_end_rgb_sec',
            'rgb_duration_frames'
        ]

    def fit(self, X, y=None):
        X_numeric = X.drop(columns=self.drop_cols, errors='ignore').select_dtypes(include=[np.number])
        
        self.sub_cols = [c for c in X_numeric.columns if 'Sub_' in c or 'roi_' in c]
        X_pca_input = X_numeric.drop(columns=self.sub_cols, errors='ignore')
        
        if not X_pca_input.empty:
            self.pca.fit(X_pca_input)
        return self

    def transform(self, X):
        X_numeric = X.drop(columns=self.drop_cols, errors='ignore').select_dtypes(include=[np.number])
        X_pca_input = X_numeric.drop(columns=self.sub_cols, errors='ignore')
        
        if not X_pca_input.empty:
            X_pca = self.pca.transform(X_pca_input)
            df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
        else:
            df_pca = pd.DataFrame()

        sub_df = X.filter(regex='Sub_|roi_').reset_index(drop=True)
        logger.info(f'[feature_extractor]: Component reduction complete. Remainder {df_pca.shape[1]} factors.')
        return pd.concat([df_pca, sub_df], axis=1)

## 4. Pipeline Execution

In [9]:
pipeline = Pipeline([
    ('constructor', feature_constructor()),
    ('aspect_ratio', aspect_ratio_transformer()),       
    ('roi_occupancy', roi_occupancy_transformer()),
    ('entity_prox', entity_interaction_transformer()),
    ('normalizer', normalize_transformer()),
    ('extractor', feature_extractor(n_comp=0.95))
])

# Execute the complete end-to-end transformation stream
df_transformed = pipeline.fit_transform(df)

print(f'Cleaned/Transfomed DataFrame Shape: {df_transformed.shape}')
print(f'Columns:\n{df_transformed.columns.tolist()}\n')
print(f'Digital Twin Features Preview:\n')
print(df_transformed.head())

2026-05-17 16:52:09.114 | INFO     | __main__:transform:54 - [feature_constructor]: Scalable camera_proximity_index and lighting_stability_index generated.
2026-05-17 16:52:09.123 | INFO     | __main__:transform:19 - [aspect_ratio_transformer]: Aspect ratio proxy for fall detection generated.
2026-05-17 16:52:09.130 | INFO     | __main__:transform:19 - [roi_occupancy_transformer]: Presence indicator at Regions of Interest (ROI) generated.
2026-05-17 16:52:09.135 | INFO     | __main__:transform:17 - [entity_interaction_transformer]: Proxies for tool proximity generated.
C:\Users\landy\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
2026-05-17 16:52:09.310 | INFO     | __main__:transform:31 - [normalize_transformer]: Stabilized distributions via Yeo-Johnson and RobustScaler over: ['Duration_sec', 'mov_intensity', 'camera_proximity_index', 'lighting_stabili

Cleaned/Transfomed DataFrame Shape: (5303, 19)
Columns:
['PC1', 'PC2', 'Sub_P02', 'Sub_P03', 'Sub_P04', 'Sub_P05', 'Sub_P06', 'Sub_P07', 'Sub_P08', 'Sub_P09', 'Sub_P10', 'Sub_P11', 'Sub_P12', 'Sub_P13', 'Sub_P14', 'Sub_P15', 'Sub_P16', 'roi_assembly_zone', 'roi_transit_zone']

Digital Twin Features Preview:

        PC1       PC2  Sub_P02  Sub_P03  Sub_P04  Sub_P05  Sub_P06  Sub_P07  \
0 -5.619978 -0.170803    False    False    False    False    False    False   
1 -0.615766  1.593960    False    False    False    False    False    False   
2  4.380584  0.035845    False    False    False    False    False    False   
3 -1.621266 -2.227129    False    False    False    False    False    False   
4  4.378903 -2.007624    False    False    False    False    False    False   

   Sub_P08  Sub_P09  Sub_P10  Sub_P11  Sub_P12  Sub_P13  Sub_P14  Sub_P15  \
0    False    False    False    False    False    False    False    False   
1    False    False    False    False    False    False    Fa

## 5. Conclusiones

Con este pipeline de ingeniería de características no solo estamos limpiando datos para cumplir con la metodología CRISP-ML(Q), sentamos las bases de la arquitectura que implementaremos en nuestro proyecto para cuando el modelo tenga que procesar video en tiempo real del piso de producción en una planta de manufactura. 

- Distancia de las cámaras (camera_proximity_index): Acorde a nuestro EDA e investigación, observamos que la escala de las cosas cambia según dónde se sitúen las cámaras en el piso de producción. Con el índice que generamos durante esta fase, en lugar de medir píxeles crudos, normalizamos la intensidad del movimiento contra la duración del clip. De esta manera, no nos impacta si el operador trabaja cerca del lente o al fondo de la estación; el modelo va a medir su velocidad de ensamble con el mismo estándar métrico.

- Estabilidad de la iluminación (lighting_stability_index): En el piso de producción la luz puede variar entre turnos o por sombras de la misma maquinaria. Al cruzar el tiempo de los sensores con los fotogramas del video RGB, creamos un índice que avisará si el tracker de YOLO está "parpadeando". Si la luz falla y se pierden detecciones, este índice lo absorbe para que el modelo predictivo no tire falsas alarmas.

- Fallos en la red, pérdida de fotogramas: InHARD es un dataset perfecto, pero en las cámaras reales con red RTSP vamos a sufrir pérdida de paquetes. Si el conteo de frames nos llega en cero o nulo por puro lag, la interpolación matemática que implementamos con SciPy nos apoyará resolver estos "huecos" en timpo real. El pipeline auto-rellena la secuencia para que al modelo nunca le llegue una señal a medias.

- Estandarización de ritmo de trabajo por operario: En nuestro EDA, durante el Avance 1, observamos que hay operadores más veloces que otros. Con Z-score  por grupo, el pipeline evalúa a cada operador contra su propio promedio histórico. Así evitamos que el sistema catalogue como "cuello de botella" o "tiempo muerto" el ritmo normal de un operario más lento por naturaleza.

- Reducción de Dimensionalidades (PCA): Incorporamos todos los pasos anteriores dentro de un Pipeline de Scikit-Learn para evitar Data Leakage al validar el modelo. Además, corregimos el error común de meter las banderas binarias (Sub_Subject) al PCA. Al aislarlas, el PCA se enfoca al 100% en exprimir la colinealidad de los datos continuos (tiempos y velocidades) al 95% de varianza, y los dummies de identidad se quedan limpios al final para no alterar la estadística.